# Kirby: reproduce the baseline and verify rollback
Copy these cells into your **existing Colab notebook/runtime**, then run in order. They reuse its worker, dataset, and cached weights. If that runtime has been deleted, rebuild the baseline with `kirby_colab.ipynb` first. No Astra key is needed yet.

This tests execution, not useful model repair. It performs one fresh 256-question baseline, compares every prediction, then runs a four-question controlled edit/restore probe. The probe hashes every model parameter before and after; hashing may take longer than inference. No checkpoint is saved or released.


## 1. Update the current workspace
Upload the latest `kirby_colab.zip` when prompted. This restores all variables needed by later cells and tests the updated backend. If multiple workspaces exist, set `BACKEND_PATH` explicitly. A process restart does not resume an interrupted edit; the next worker loads the original checkpoint.


In [ ]:
from google.colab import files, userdata
from pathlib import Path, PurePosixPath
import hashlib, io, json, os, re, signal, stat, subprocess, sys, uuid, zipfile

# If several workspaces exist, paste the intended backend path here.
BACKEND_PATH = ""
WORKER_PYTHON = ""
if BACKEND_PATH:
    backend = Path(BACKEND_PATH)
else:
    candidates = sorted(Path("/content").glob("kirby-*/backend"))
    candidates = [p for p in candidates if (p / "data" / "plan-dev-256" / "development.json").is_file()]
    if len(candidates) != 1:
        raise RuntimeError("Set BACKEND_PATH to the backend with your 256-question plan. Found: "
                           + str([str(p) for p in candidates])
                           + ". If the runtime was reset, use the main notebook to rebuild the baseline first.")
    backend = candidates[0]
workspace = backend.parent
if not (backend / "data" / "plan-dev-256" / "development.json").is_file():
    raise RuntimeError("This workspace has no 256-question development plan.")
workers = sorted(workspace.glob("worker-venv-*/bin/python"),
                 key=lambda p: p.parent.parent.stat().st_mtime)
worker_python = WORKER_PYTHON or (str(workers[-1]) if workers else "")
if not worker_python or not Path(worker_python).is_file():
    raise RuntimeError("Worker environment missing. Run setup in the main notebook first.")
print("Updating:", backend)
bundle_bytes = globals().pop("KIRBY_BUNDLE_BYTES", None)
if bundle_bytes is None:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload only the newly built kirby_colab.zip.")
    bundle_bytes = next(iter(uploaded.values()))
    del uploaded
with zipfile.ZipFile(io.BytesIO(bundle_bytes)) as archive:
    members = archive.infolist()
    if len(members) > 100 or sum(m.file_size for m in members) > 5_000_000:
        raise ValueError("Unexpected source bundle size.")
    replacements = {}
    for member in members:
        name = PurePosixPath(member.filename)
        allowed = (str(name) in {"backend/README.md", "backend/pyproject.toml"} or
                   (len(name.parts) == 3 and name.parts[0] == "backend" and
                    ((name.parts[1] in {"kirby", "tests"} and name.suffix == ".py") or
                     (name.parts[1] == "notebooks" and name.suffix == ".ipynb") or
                     (name.parts[1] == "docs" and name.suffix == ".md"))))
        if (not allowed or ".." in name.parts or name.is_absolute() or
                "\\" in member.filename or ":" in member.filename or
                stat.S_ISLNK(member.external_attr >> 16) or member.filename in replacements):
            raise ValueError("Unsafe or unexpected source bundle member.")
        replacements[member.filename] = archive.read(member)
    required = {"backend/kirby/patches.py", "backend/kirby/rollback.py", "backend/kirby/reproducibility.py"}
    if not required.issubset(replacements):
        raise ValueError("This ZIP predates the rollback implementation.")
for name, payload in replacements.items():
    target = workspace / name
    if target.is_symlink() or not target.resolve().is_relative_to(backend.resolve()):
        raise ValueError("Source destination leaves the backend workspace.")
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(payload)
del bundle_bytes, replacements

def run_setup(command, **kwargs):
    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, **kwargs)
    print(result.stdout, end="")
    result.check_returncode()
    return result

def run_worker(command):
    process = subprocess.Popen(command, cwd=backend, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        code = process.wait()
    except KeyboardInterrupt:
        process.send_signal(signal.SIGINT)
        try:
            remaining, _ = process.communicate(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill()
            remaining, _ = process.communicate()
        print(remaining, end="")
        raise
    finally:
        process.stdout.close()
    if code:
        raise subprocess.CalledProcessError(code, command)

MODEL_ID = "johnsnowlabs/JSL-MedPhi2-2.7B"
MODEL_REVISION = "b313a242eb211863f7b62624b345b321a3b114ea"
PRECISION = "float32"  # Matches your T4 baseline.
MAX_SECONDS = 3600
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("HF_HOME", "/content/kirby-hf-cache")
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
revision = MODEL_REVISION
(workspace / "model-revision.txt").write_text(revision, encoding="utf-8")
plan = backend / "data" / "plan-dev-256"
runs = backend / "runs"
run_setup([worker_python, "-c", "import torch; from transformers import PhiConfig, PhiForCausalLM; assert torch.cuda.is_available(), 'Select a GPU runtime'; print(torch.cuda.get_device_name(0))"], cwd=backend)
run_setup([worker_python, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=backend)
run_setup([worker_python, "-m", "kirby", "validate", "--manifest", str(plan / "development.json")], cwd=backend)
print("Ready. Dataset and existing evidence retained.")


## 2. Repeat and compare the unchanged baseline
The latest completed 256-question run is selected by finish time, never accuracy; set `REFERENCE_RUN` to select it explicitly. Comparison requires identical evaluation code, protocol, dataset, model identity, generation settings, environment, and predictions. Latency is allowed to vary. If the prior code/environment differs, this cell saves the mismatch and stops. Review it, then rerun this cell to compare a fresh pair under the updated setup. Do not proceed through a prediction mismatch.


In [ ]:
# Optional: choose an explicit completed 256-question reference run directory.
REFERENCE_RUN = ""
if REFERENCE_RUN:
    reference_run = Path(REFERENCE_RUN)
else:
    candidates = []
    for path in runs.glob("*/report.json"):
        saved = json.loads(path.read_text(encoding="utf-8"))
        if (saved.get("task") == "evaluate_model" and saved.get("state") == "completed"
                and saved.get("development", {}).get("total") == 256):
            candidates.append((saved["finished_at"], path.parent))
    if not candidates:
        raise RuntimeError("No completed 256-question baseline found. Run Step 6 of the main notebook first.")
    reference_run = max(candidates, key=lambda item: item[0])[1]
print("Reference run:", reference_run)
before = set(runs.glob("*/report.json"))
common = ["--manifest", str(plan / "development.json"), "--model", MODEL_ID,
          "--revision", MODEL_REVISION, "--precision", PRECISION,
          "--allow-download", "--max-seconds", str(MAX_SECONDS), "--output", str(runs)]
run_worker([worker_python, "-u", "-m", "kirby", "baseline", *common])
created = set(runs.glob("*/report.json")) - before
if len(created) != 1:
    raise RuntimeError("Expected one new baseline report.")
repeat_run = created.pop().parent
comparison_path = runs / ("reproducibility-" + str(uuid.uuid4())) / "comparison.json"
run_worker([worker_python, "-u", "-m", "kirby", "compare-baselines",
            "--first", str(reference_run), "--second", str(repeat_run),
            "--output", str(comparison_path)])
comparison = json.loads(comparison_path.read_text(encoding="utf-8"))
print("Reproduced:", comparison["reproduced"], "| evidence:", comparison_path)


## 3. Apply a controlled edit and verify exact rollback
The worker scales column 0 of layer 0's Phi MLP output projection by 0.97. It rejects nonfinite/no-op/excessive edits, scores the same four development questions before/during/after, restores the original bytes even after an inference exception, and verifies complete weight hashes plus restored outputs. The edited answers need not differ for a valid execution check. Never describe a score increase here as Astra repair. On failure, download the evidence; do not weaken checks to obtain a passing result.


In [ ]:
if "comparison_path" not in globals() or not comparison_path.is_file():
    raise RuntimeError("Run the baseline comparison cell first.")
comparison = json.loads(comparison_path.read_text(encoding="utf-8"))
if not comparison.get("reproduced"):
    raise RuntimeError("Baseline reproducibility has not passed; inspect comparison.json first.")
for reference in comparison["runs"]:
    for filename, field in (("report.json", "report_sha256"), ("predictions.jsonl", "predictions_sha256")):
        payload = (Path(reference["directory"]) / filename).read_bytes()
        if hashlib.sha256(payload).hexdigest() != reference[field]:
            raise RuntimeError("Compared run evidence changed; repeat the comparison.")

# Fixed execution probe, chosen before seeing edited-model performance.
# This is not an optimization attempt and is never retained.
before = set(runs.glob("*/report.json"))
run_worker([worker_python, "-u", "-m", "kirby", "rollback-check",
            "--manifest", str(plan / "development.json"), "--model", MODEL_ID,
            "--revision", MODEL_REVISION, "--precision", PRECISION,
            "--allow-download", "--max-seconds", str(MAX_SECONDS), "--output", str(runs),
            "--questions", "4", "--layer", "0", "--column", "0", "--scale", "0.97"])
created = set(runs.glob("*/report.json")) - before
if len(created) != 1:
    raise RuntimeError("Expected one rollback report. Inspect runs and download the evidence.")
rollback_report_path = created.pop()
rollback_report = json.loads(rollback_report_path.read_text(encoding="utf-8"))
print(json.dumps({key: rollback_report.get(key) for key in
                 ("state", "outcome", "weights_restored", "outputs_restored", "patch", "stages", "error")}, indent=2))
print("Rollback evidence:", rollback_report_path)


## 4. Download evidence
Run this even after a failure. It includes the comparison, rollback receipt, original/edited/restored predictions, and all baseline reports. Model weights and raw datasets are excluded.


In [ ]:
from datetime import datetime, timezone
inventory = subprocess.check_output([worker_python, '-m', 'pip', 'list', '--format=json'], text=True)
(workspace / 'package-versions.json').write_text(inventory, encoding='utf-8')
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
destination = workspace / f'kirby-evidence-{stamp}.zip'
with zipfile.ZipFile(destination, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for source in sorted((backend / 'runs').glob('*/*')):
        if source.is_file() and source.suffix in {'.json', '.jsonl', '.md'}:
            archive.write(source, str(source.relative_to(workspace)))
    for source in sorted((backend / 'data').glob('plan-dev-*/*.json')):
        archive.write(source, str(source.relative_to(workspace)))
    for source in sorted((backend / 'kirby').glob('*.py')):
        archive.write(source, str(source.relative_to(workspace)))
    for source in (backend / 'pyproject.toml', workspace / 'package-versions.json', workspace / 'model-revision.txt'):
        if source.is_file():
            archive.write(source, str(source.relative_to(workspace)))
print('Downloading:', destination.name)
files.download(str(destination))


## Next milestone
A completed report must say `Controlled rollback verified`, with both `weights_restored` and `outputs_restored` true. Then continue with `kirby_astra_colab.ipynb` for one implemented Astra proposal. Cumulative accepted edits, regression evaluation, and export/reload verification remain separate work.
